In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager  
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from time import sleep
from bs4 import BeautifulSoup
import pandas as pd
import random
import re


Hàm lấy số trang trong trang tìm kiếm


In [13]:
def Get_num_page():
    num_page = driver.find_element(By.ID,'job-listing-paginate-text').text
    parts = num_page.split('/')
    num  = parts[1].split(' ')
    Total_page = int(num[1])
    return Total_page

Hàm lấy từng link này 

In [14]:
def get_link():
    els = driver.find_elements(By.CSS_SELECTOR, "a[href*='/viec-lam/']")
    sleep(random.uniform(1, 3)) 
    links = set()
    for e in els:
        href = e.get_attribute("href")
        links.add(href)

    All_link = list(links)
    return All_link


Sao đó crawl từng ô từng link nghề nghiệp theo định dạng dưới đây

In [15]:
def Get_data(Link):
    driver.get(Link)
    data = {}
    page_source = BeautifulSoup(driver.page_source, "html.parser")
    # lấy góc phần tư thứ nhất

    # nơi chứu title
    Title_Class = page_source.find('h1','a', class_ = 'job-detail__info--title').get_text(strip=True,)
    data['Tên tiêu đề'] = Title_Class

    #tên cônng ty
    company_name = page_source.find('a',class_= 'name').get_text(strip= True)
    data['Tên công ty'] = company_name

    # nơi chứa lương địa điểm, số năm kinh nghiệm
    Section_list = []
    Section_Class = page_source.find_all('div', class_ ='job-detail__info--section-content-value')
    for Section in Section_Class:
        Section.find('div')
        Section_list.append(Section.get_text())
    data["Lương"] = Section_list[0]
    data ["Thành Phố"] = Section_list[1]
    data ["Kinh nghiệm"] = Section_list[2]

    #deadline
    deadline_Class = page_source.find('div', class_ ='job-detail__info--deadline').getText(strip=True)
    dealdine = deadline_Class.split(':')
    data['Ngày hết hạn']=dealdine[1].strip()

    # nơi lưu tag
    tags_field = page_source.find('div','a', class_='job-tags').get_text()
    tags = tags_field.split('\n')
    for tag in tags:
        if tag == "":
            tags.remove(tag)
    data['Tags'] = tags 


    # nơi lưu thông tin chung

    infor_jobs = page_source.find_all('div', class_='box-general-group-info-value')
    infor_list = []
    for infor in infor_jobs:
        infor.find('div')
        infor_list.append(infor.get_text())
    data["Chức vụ"] = infor_list[0]
    data ["Học Vấn"] = infor_list[1]
    data ["Số lượng"] = infor_list[2]
    data["Hình thức"] =infor_list[3]

    # Kỹ năng cần có


    skills = page_source.find_all('span',class_ = 'box-category-tag')
    if(skills):
        skills_list = []

        for skill in skills:
            skills_list.append(skill.get_text())
        data["Kỹ năng"] = skills_list
    else:
        data['Kỹ năng'] =''
    # nơi lưu mô tả

    # 1) Tạo khung biến để lưu theo từng mục
    data['mo_ta_cong_viec'] = []
    data['yeu_cau_ung_vien'] = []
    data['quyen_loi'] = []
    data['dia_diem_lam_viec'] = []
    data['thoi_gian_lam_viec'] = []


    # 2) Duyệt từng block .job-description__item
    for item in page_source.select(".job-description__item"):
        h3 = item.find("h3")
        content = item.select_one(".job-description__item--content")
        if not h3 or not content:
            continue

        title = h3.get_text(" ", strip=True).lower()

        # 3) Lấy text theo từng dòng trong block hiện tại (chỉ cho block này)
        lines = []
        for tag in content.find_all(["p","li","div"], recursive=True):
            # tránh nhân đôi khi <div> bao <p>/<li>
            if tag.name == "div" and tag.find(["p","li"]):
                continue
            txt = tag.get_text(" ", strip=True)
            if not txt:
                continue
            # bỏ ký tự đầu dòng +, -, •
            txt = re.sub(r"^\s*([+\-•]|–)\s*", "", txt)
            #^ = bắt đầu dòng
            # \s* = khoảng trắng bất kỳ
            # ([+\-•]|–) = nếu có dấu +, -, • hoặc gạch ngang –
            # \s* = thêm khoảng trắng sau đó
            # → thì xóa đi (thay bằng rỗng "").
            # gom khoảng trắng
            txt = re.sub(r"\s+", " ", txt).strip()
            if txt:
                lines.append(txt)

        # fallback nếu block không có p/li/div con
        if not lines:
            txt = content.get_text(" ", strip=True)
            if txt:
                lines = [re.sub(r"\s+"," ",txt).strip()] 

        # 4) Gán đúng “bậc class” theo tiêu đề h3
        if "mô tả công việc" in title or title.strip() == "mô tả":
            data['mo_ta_cong_viec'].extend(lines)
        elif "yêu cầu" in title:
            data['yeu_cau_ung_vien'].extend(lines)
        elif "quyền lợi" in title:
            data['quyen_loi'].extend(lines)
        elif "địa điểm làm việc" in title:
            data['dia_diem_lam_viec'].extend(lines)
        elif "thời gian làm việc" in title:
            data['thoi_gian_lam_viec'].extend(lines)
    data['url'] = Link
    return data



In [16]:
link = ''

In [17]:
url = "https://www.topcv.vn/"
service = ChromeService(executable_path=r"D:\Others\BrowserDriver\chromedriver-win64\chromedriver.exe")
driver = webdriver.Chrome(service=service)
driver.get(url)
# truy cập web

filter_IT = driver.find_element(By.XPATH,'//*[@id="category-filter-navigation-home-desktop-container"]/div/div[1]/div[5]/a')
filter_IT.click()
# sử dụng nút lọc có sẵn ở web


In [18]:
All_links = []
Link_not_add = []

Total_page = Get_num_page()

for i in range(1, Total_page + 1):
    try:
        # Lấy link trên trang hiện tại
        WebDriverWait(driver,2)
        links = get_link()
        if links:
            All_links.extend(links)
            print(f"Crawl thành công trang {i}/{Total_page}, thu được {len(links)} link")
        else:
            Link_not_add.append(driver.current_url)
            print(f"Không lấy được link ở trang {i}")

        # Nếu chưa phải trang cuối thì bấm nút Next
        if i < Total_page:
            try:
                # Cuộn dần dần xuống trước
                for step in range(3):
                    driver.execute_script("window.scrollBy(0, 400);")
                    sleep(random.uniform(1.8, 3))

                # Tìm và scroll nút Next vào giữa màn hình
                next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
                sleep(random.uniform(3, 5))

                # Click Next
                next_button.click()
                sleep(random.uniform(2, 5))

            except Exception as e:
                print(f"Không bấm được Next ở trang {i}: {e}")
                Link_not_add.append(driver.current_url)
                break


    except Exception as error:
        print(f"Lỗi ở trang {i}: {error}")
        Link_not_add.append(driver.current_url)
        continue

print("\n=== KẾT QUẢ ===")
print(f"Tổng số link thu được: {len(All_links)}")
print(f"Số trang lỗi: {len(Link_not_add)}")


Crawl thành công trang 1/70, thu được 49 link
Crawl thành công trang 2/70, thu được 1 link
Crawl thành công trang 3/70, thu được 1 link
Không bấm được Next ở trang 3: Message: no such element: Unable to locate element: {"method":"css selector","selector":"a[rel='next']"}
  (Session info: chrome=140.0.7339.80); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	GetHandleVerifier [0x0x7ff6360f6b25+79621]
	GetHandleVerifier [0x0x7ff6360f6b80+79712]
	(No symbol) [0x0x7ff635e8c0ea]
	(No symbol) [0x0x7ff635ee2f56]
	(No symbol) [0x0x7ff635ee320c]
	(No symbol) [0x0x7ff635f365b7]
	(No symbol) [0x0x7ff635f0b17f]
	(No symbol) [0x0x7ff635f333d0]
	(No symbol) [0x0x7ff635f0af13]
	(No symbol) [0x0x7ff635ed4151]
	(No symbol) [0x0x7ff635ed4ee3]
	GetHandleVerifier [0x0x7ff6363b683d+2962461]
	GetHandleVerifier [0x0x7ff6363b0b5d+2938685]
	GetHandleVerifier [0x0x7ff6363cf71d+3064573]
	GetHandleVerifier [

In [19]:
print("Số link:", len(All_links))
df = pd.DataFrame(All_links)
df.to_csv("TopCV_links.csv")

Số link: 51


In [20]:
print(len(All_links))

51


Hàm chính crawl data

In [21]:
data_jobs = []
link_error = []
i = 0
for link in All_links:
    try:
        sleep(random.uniform(2.8,5))
        data = Get_data(link)
        i+=1
        print(f'{i}/{len(All_links)}')
        if data:
            data_jobs.append(data)    
            print(f"Crawl Thành thông {i}",end = None) 

        if ( i%10 ==0 ):
            sleep(random.randint(10,15))
            a = i//5
            # quay lại trang nào đó để có thể ko bị chặn
            driver.get(f"https://www.topcv.vn/tim-viec-lam-cong-nghe-thong-tin-cr257?type_keyword=1&page={a}&category_family=r257")
    except Exception as error:
        print(f"lỗi:{error} \n{link}")
        sleep(random.randint(15,20))
        link_error.append(link) 
        print(f"lỗi ")
        continue
    

lỗi:'NoneType' object has no attribute 'get_text' 
https://www.topcv.vn/viec-lam/senior-front-end-developer-angular/1857422.html?ta_source=JobSearchList_LinkDetail&u_sr_id=YWpSbfmlPWOW1w69P7dI1CEAad69zk072Ww88HEW_1757320909
lỗi 
1/51
Crawl Thành thông 1
2/51
Crawl Thành thông 2
3/51
Crawl Thành thông 3
4/51
Crawl Thành thông 4
5/51
Crawl Thành thông 5
6/51
Crawl Thành thông 6
7/51
Crawl Thành thông 7
8/51
Crawl Thành thông 8
9/51
Crawl Thành thông 9
10/51
Crawl Thành thông 10
11/51
Crawl Thành thông 11
12/51
Crawl Thành thông 12
13/51
Crawl Thành thông 13
14/51
Crawl Thành thông 14
15/51
Crawl Thành thông 15
16/51
Crawl Thành thông 16
17/51
Crawl Thành thông 17
18/51
Crawl Thành thông 18
19/51
Crawl Thành thông 19
20/51
Crawl Thành thông 20
lỗi:'NoneType' object has no attribute 'get_text' 
https://www.topcv.vn/viec-lam/project-manager/1854862.html?ta_source=JobSearchList_LinkDetail&u_sr_id=YWpSbfmlPWOW1w69P7dI1CEAad69zk072Ww88HEW_1757320909
lỗi 
lỗi:'NoneType' object has no attribute 

coi giá liệu trước khi gắn

In [22]:
df = pd.DataFrame(data_jobs)
df.tail(5)

,Tên tiêu đề,Tên công ty,Lương,Thành Phố,Kinh nghiệm,Ngày hết hạn,Tags,Chức vụ,Học Vấn,Số lượng,Hình thức,Kỹ năng,mo_ta_cong_viec,yeu_cau_ung_vien,quyen_loi,dia_diem_lam_viec,thoi_gian_lam_viec,url
28,Business Analyst(English Fluently),Công ty Cổ phần MISA,18 - 30 triệu,Hà Nội,2 năm,04/10/2025,[Chuyên môn Business Analyst (Phân tích nghiệp...,Nhân viên,Đại Học trở lên,5 người,Toàn thời gian,"[Phân Tích Nghiệp Vụ, Giao Tiếp (tiếng Anh), T...","[About MISA For nearly 30 years, MISA has been...",[A Bachelor’s degree or higher in Business Adm...,"[Joining MISA means being part of a dynamic, f...","[Hà Nội: Tòa nhà Technosoft, ngõ 15 phố Duy Tâ...",[Thứ 2 - Thứ 6 (từ 08:00 đến 17:30)],https://www.topcv.vn/viec-lam/business-analyst...
29,Talented Developers (C++ / C#),KOH YOUNG TECHNOLOGY INC. VIETNAM,Thoả thuận,Hà Nội,3 năm,29/09/2025,"[Chuyên môn Software Engineer, IT - Phần mềm]",Nhân viên,Đại Học trở lên,10 người,Toàn thời gian,"[3D Artist, SQL Server, Thành thạo các ngôn ng...",[We are looking for passionate software develo...,"[C++ Developer, 3+ years of hands-on experienc...",[You’ll be part of a global leader in 3D inspe...,"[Hà Nội: Tầng 17, Tòa nhà Discovery 302 Cầu Gi...",[],https://www.topcv.vn/viec-lam/talented-develop...
30,Nhân Viên Kinh DoanhThu Nhập Đến 20 Triệu (Đầy...,CÔNG TY TNHH DE-LIFESTYLE SERVICES,22 - 33 triệu,Hồ Chí Minh,3 năm,25/09/2025,"[Chuyên môn Sales IT Phần mềm khác, B2C, Có hỗ...",Nhân viên,Cao Đẳng trở lên,1 người,Toàn thời gian,,"[1. Mảng Cleaning Service, Tìm kiếm và phát tr...",[Tốt nghiệp Cao đẳng/Đại học chuyên ngành Kinh...,"[Chúng tôi cung cấp gói đãi ngộ cạnh tranh, ba...",[Hồ Chí Minh: Quận 2],[Thứ 2 - Thứ 6 (từ 08:00 đến 17:30)],https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...
31,IOS DeveloperNative (Swift/C),CÔNG TY TNHH PHẦN MỀM TOWER HÀ NỘI,Thoả thuận,Hà Nội,1 năm,30/09/2025,"[Chuyên môn Mobile Developer, IT - Phần mềm]",Nhân viên,Đại Học trở lên,2 người,Toàn thời gian,"[Objective-C, Swift, Database, API, Problem-so...","[Phát triển các ứng dụng iOS sử dụng Swift, Ti...",[Yêu cầu bắt buộc: • Tốt nghiệp ĐH/CĐ chuyên n...,[Chế độ đãi ngộ: - Lương đàm phán theo chuyên ...,"[Hà Nội: Tòa nhà Viwaseen 48 đường Tố Hữu, phư...","[Thứ 2 - Thứ 6 (từ 08:00 đến 17:30), Sáng thứ ...",https://www.topcv.vn/viec-lam/ios-developer-na...
32,Customer Service Specialist(English/Korean/Man...,Tek Experts Vietnam,14 - 26 triệu,Hà Nội,2 năm,03/10/2025,[Chuyên môn Dịch vụ khách hàng/Chăm sóc khách ...,Nhân viên,Đại Học trở lên,3 người,Toàn thời gian,,"[Resolution Customer, Acts as a primary contac...","[Bachelor's degree in technology, business, or...","[Competitive salary package, Full salary provi...","[Hà Nội: Lotte Center, Ba Đình]",[],https://www.topcv.vn/viec-lam/customer-service...


Thả vào file csv

In [23]:
df = pd.DataFrame(data_jobs)
df.to_csv("data_topCV.csv", header=True, index=False, encoding="utf-8-sig")

Lấy các link ko crawl được để crawl lại

In [24]:
E_links = pd.DataFrame(link_error)
E_links.to_csv('Error_links.csv',mode="a", header=False, index=False)

In [25]:
driver.quit()